In [ ]:
# =========================================
# TIME SERIES LEARNING LAB - Cornelius-Snowhyt ❄️
# Fully Enhanced with Rolling Avg, ACF, and OLS CI
# =========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf
from sklearn.linear_model import LinearRegression
from pandas.tseries.offsets import MonthEnd

# -----------------------------
# PART 1: ELECTRICITY USAGE VS TEMPERATURE
# -----------------------------
df_elec = pd.DataFrame({
    "Temperature": [30, 32, 35, 40, 45, 50, 48, 45, 40, 35, 32, 30],
    "Electricity_Usage": [200, 210, 230, 300, 400, 420, 410, 390, 350, 300, 250, 220]
})

# --- Raw Scatter Plot ---
fig, ax = plt.subplots(figsize=(10,5))
ax.scatter(df_elec["Temperature"], df_elec["Electricity_Usage"], color='blue', label='Observed')
ax.set_title("Electricity Usage vs Temperature — Cornelius-Snowhyt ❄️")
ax.set_xlabel("Temperature (°C)")
ax.set_ylabel("Electricity Usage (kWh)")
ax.legend()
plt.show()

# --- OLS Regression & Prediction ---
X = df_elec[["Temperature"]]
y = df_elec["Electricity_Usage"]
model_elec = LinearRegression()
model_elec.fit(X, y)
temp_predict = 42
usage_predict = model_elec.predict([[temp_predict]])[0]

# --- Plot Regression + Prediction ---
fig, ax = plt.subplots(figsize=(10,5))
ax.scatter(df_elec["Temperature"], df_elec["Electricity_Usage"], color='blue', label='Observed')
ax.plot(df_elec["Temperature"], model_elec.predict(X), color='red', label='OLS Linear Trend')
ax.scatter(temp_predict, usage_predict, color='green', s=100, label=f'Prediction at {temp_predict}°C')
ax.set_title("Electricity Usage with OLS Trend & Prediction — Cornelius-Snowhyt ❄️")
ax.set_xlabel("Temperature (°C)")
ax.set_ylabel("Electricity Usage (kWh)")
ax.legend()
plt.show()

# --- OLS Prediction with Confidence Interval ---
X_const = sm.add_constant(df_elec["Temperature"])
ols_model = sm.OLS(df_elec["Electricity_Usage"], X_const).fit()
pred_summary = ols_model.get_prediction([1, temp_predict]).summary_frame(alpha=0.05)
print("OLS Prediction at 42°C with 95% CI:")
print(pred_summary)

# -----------------------------
# PART 2: AIRLINE PASSENGERS TIME SERIES
# -----------------------------
air = pd.read_csv("air_passengers.csv")
air.columns = air.columns.str.strip().str.replace("#","")
air["Month"] = pd.to_datetime(air["Month"])
air.set_index("Month", inplace=True)

# --- Raw Plot ---
fig, ax = plt.subplots(figsize=(12,5))
ax.plot(air.index, air["Passengers"], marker='o', linestyle='-', color='blue')
ax.set_title("Monthly Airline Passengers — Raw Data (Cornelius-Snowhyt ❄️)")
ax.set_xlabel("Month")
ax.set_ylabel("Passengers")
plt.show()

# --- OLS Linear Trend ---
air["t"] = np.arange(len(air))
X_air = sm.add_constant(air["t"])
y_air = air["Passengers"]
ols_airline = sm.OLS(y_air, X_air).fit()
air["trend"] = ols_airline.predict(X_air)

fig, ax = plt.subplots(figsize=(12,5))
ax.plot(air.index, air["Passengers"], marker='o', label='Observed')
ax.plot(air.index, air["trend"], color='red', label='OLS Linear Trend')
ax.set_title("Airline Passengers with Linear Trend — Cornelius-Snowhyt ❄️")
ax.set_xlabel("Month")
ax.set_ylabel("Passengers")
ax.legend()
plt.show()

# --- Decomposition ---
decomp = seasonal_decompose(air["Passengers"], model='additive', period=12)
fig, ax = plt.subplots(4,1, figsize=(12,10), sharex=True)
ax[0].plot(air.index, decomp.observed)
ax[0].set_title("Observed")
ax[1].plot(air.index, decomp.trend)
ax[1].set_title("Trend")
ax[2].plot(air.index, decomp.seasonal)
ax[2].set_title("Seasonality")
ax[3].plot(air.index, decomp.resid)
ax[3].set_title("Residuals")
plt.tight_layout()
plt.show()

# --- Rolling Average (12-month) ---
air['Passengers_Roll12'] = air['Passengers'].rolling(window=12).mean()
fig, ax = plt.subplots(figsize=(12,5))
ax.plot(air.index, air['Passengers'], label='Observed')
ax.plot(air.index, air['Passengers_Roll12'], color='orange', label='12-Month Rolling Avg')
ax.set_title("Rolling Average of Passengers — Cornelius-Snowhyt ❄️")
ax.legend()
plt.show()

# --- ARIMA Forecast ---
arima_model = ARIMA(air["Passengers"], order=(1,1,1))
arima_fit = arima_model.fit()
forecast_arima = arima_fit.forecast(12)
last_month = air.index[-1]
forecast_index = pd.date_range(start=last_month + MonthEnd(1), periods=12, freq='M')

fig, ax = plt.subplots(figsize=(12,5))
ax.plot(air.index, air["Passengers"], label='Observed')
ax.plot(forecast_index, forecast_arima, color='red', label='ARIMA Forecast')
ax.set_title("ARIMA Forecast — Cornelius-Snowhyt ❄️")
ax.set_xlabel("Month")
ax.set_ylabel("Passengers")
ax.legend()
plt.show()

# --- ACF of ARIMA Residuals ---
residuals = arima_fit.resid
fig, ax = plt.subplots(figsize=(10,4))
plot_acf(residuals, ax=ax, lags=20)
ax.set_title("ACF of ARIMA Residuals — Cornelius-Snowhyt ❄️")
plt.show()

# --- SARIMA Forecast ---
sarima_model = SARIMAX(air["Passengers"], order=(1,1,1), seasonal_order=(1,1,1,12))
sarima_fit = sarima_model.fit(disp=False)
forecast_sarima = sarima_fit.get_forecast(12)
forecast_mean = forecast_sarima.predicted_mean
forecast_ci = forecast_sarima.conf_int()

fig, ax = plt.subplots(figsize=(12,5))
ax.plot(air.index, air["Passengers"], label='Observed')
ax.plot(forecast_index, forecast_mean, color='green', label='SARIMA Forecast')
ax.fill_between(forecast_index, forecast_ci.iloc[:,0], forecast_ci.iloc[:,1], color='lightgreen', alpha=0.3)
ax.set_title("SARIMA Forecast — Cornelius-Snowhyt ❄️")
ax.set_xlabel("Month")
ax.set_ylabel("Passengers")
ax.legend()
plt.show()

print("=== Notebook Complete: Enhanced Time Series Learning Lab for Cornelius-Snowhyt ❄️ ===")